# Класифікація емоцій: від TF-IDF до дообучених трансформерів

**Курсовий проєкт з глибокого навчання**

Цей ноутбук описує повний цикл розробки класифікатора емоцій тексту на 6 класів на основі датасету **dair-ai/emotion** (~417 тис. коротких речень англійською мовою з мітками `sadness`, `joy`, `love`, `anger`, `fear`, `surprise`).

Навчено та порівняно три моделі в порядку зростання складності:

1. **Базова модель (Baseline)** — TF-IDF + логістична регресія (класична референсна точка)
2. **Глибоке навчання з нуля** — Embedding + BiLSTM, навчена з випадкової ініціалізації на PyTorch
3. **Transfer learning** — дообучений `distilbert-base-uncased` (Hugging Face)

**Середовище виконання:** Runtime → Change runtime type → GPU (достатньо T4).


In [ ]:
# 1. Налаштування — клонуємо/завантажуємо репозиторій, встановлюємо залежності
# Якщо запускаєте з GitHub-репозиторію в Colab:
!git clone https://github.com/arttikul/emotion-classification-capstone
%cd emotion-classification-capstone

!pip install -q -r requirements.txt


In [ ]:
import sys
sys.path.append("src")

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data import LABEL_NAMES, ID2LABEL, load_emotion_data, split_data
from viz import plot_confusion_matrix, plot_training_curves, plot_model_comparison

CSV_PATH = "data/emotion-dataset.csv"   # покладіть emotion-dataset.csv у data/
SAMPLE_FRAC = 1.0   # встановіть < 1.0 (напр. 0.2) для швидшого прогону під час ітерацій


## 2. Розвідувальний аналіз даних (EDA)

Датасет має 6 класів емоцій із природним дисбалансом (joy/sadness трапляються часто, surprise/love — рідше) — це впливає на те, як ми зважуємо функцію втрат і якій метриці довіряємо (macro-F1, а не лише accuracy).


In [ ]:
df = load_emotion_data(CSV_PATH, sample_frac=SAMPLE_FRAC)
print(f"Total examples: {len(df):,}")

label_counts = df["label"].map(ID2LABEL).value_counts()
print(label_counts)

fig, ax = plt.subplots(figsize=(6, 4))
label_counts.sort_index().plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Class distribution")
ax.set_ylabel("# examples")
plt.tight_layout()
plt.show()


In [ ]:
# Декілька прикладів рядків для кожного класу
for label_id, name in ID2LABEL.items():
    sample = df[df["label"] == label_id]["text"].sample(2, random_state=1).tolist()
    print(f"\n[{name}]")
    for s in sample:
        print(" -", s)


In [ ]:
train_df, val_df, test_df = split_data(df)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

train_df.to_csv("data/train.csv", index=False)
val_df.to_csv("data/val.csv", index=False)
test_df.to_csv("data/test.csv", index=False)


## 3. Базова модель — TF-IDF + логістична регресія

Сильна і швидка класична базова модель. Будь-яка модель глибокого навчання, яку ми побудуємо, має чітко перевершити цей результат (або ми маємо вміти пояснити, чому це не так).


In [ ]:
from train_baseline import run as run_baseline

baseline_metrics = run_baseline(CSV_PATH, sample_frac=SAMPLE_FRAC, out_dir="artifacts/baseline")


In [ ]:
plot_confusion_matrix(
    baseline_metrics["confusion_matrix"], LABEL_NAMES, "Baseline (TF-IDF + LogReg) — confusion matrix"
)
plt.show()


## 4. Глибоке навчання з нуля — BiLSTM

Шар ембеддингів, навчений з випадкової ініціалізації, подається на вхід 2-шаровому двонаправленому LSTM; фінальні прямий та зворотний приховані стани конкатенуються і передаються через лінійний класифікаційний шар. Це точка даних "чисте глибоке навчання, без попереднього навчання".


In [ ]:
from train_lstm import run as run_lstm

lstm_metrics = run_lstm(
    csv_path=CSV_PATH,
    sample_frac=SAMPLE_FRAC,
    epochs=6,
    batch_size=128,
    max_len=40,
    lr=1e-3,
    out_dir="artifacts/lstm",
)


In [ ]:
plot_training_curves(lstm_metrics["history"], title="BiLSTM training curves")
plt.show()

plot_confusion_matrix(lstm_metrics["confusion_matrix"], LABEL_NAMES, "BiLSTM — confusion matrix")
plt.show()


## 5. Transfer learning — дообучений DistilBERT

Дообучення попередньо навченого трансформера (`distilbert-base-uncased`) за допомогою Hugging Face `Trainer` API. Очікується, що ця модель перевершить BiLSTM, навчену з нуля, оскільки вона стартує з мовних представлень, вже вивчених на величезному текстовому корпусі, а не вчить усе заново лише на цьому датасеті.

**Примітка:** цей крок потребує GPU-середовища — на CPU навчання буде повільним.


In [ ]:
from train_transformer import run as run_transformer

transformer_metrics = run_transformer(
    csv_path=CSV_PATH,
    sample_frac=SAMPLE_FRAC,
    epochs=2,
    batch_size=32,
    max_len=64,
    lr=2e-5,
    out_dir="artifacts/transformer",
)


In [ ]:
plot_confusion_matrix(
    transformer_metrics["confusion_matrix"], LABEL_NAMES, "Fine-tuned DistilBERT — confusion matrix"
)
plt.show()


## 6. Порівняння моделей

Порівняння всіх трьох моделей поруч на одному й тому ж відкладеному тестовому наборі.


In [ ]:
results = {
    "TF-IDF + LogReg": {
        "accuracy": baseline_metrics["accuracy"],
        "macro_f1": baseline_metrics["macro_f1"],
    },
    "BiLSTM (scratch)": {
        "accuracy": lstm_metrics["test_accuracy"],
        "macro_f1": lstm_metrics["test_macro_f1"],
    },
    "DistilBERT (fine-tuned)": {
        "accuracy": transformer_metrics["test_accuracy"],
        "macro_f1": transformer_metrics["test_macro_f1"],
    },
}

comparison_df = pd.DataFrame(results).T
comparison_df.columns = ["Accuracy", "Macro-F1"]
display(comparison_df.round(4))

plot_model_comparison(results)
plt.show()


## 7. Демонстрація інференсу

Перевіримо дообучений трансформер на кількох нових, раніше не бачених реченнях.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "artifacts/transformer/final_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

samples = [
    "I can't believe I finally got the job, I'm over the moon!",
    "I'm terrified of what might happen tomorrow.",
    "This is so unfair, I'm absolutely furious about it.",
    "I miss my grandmother so much, I keep crying at night.",
    "Wait, you're kidding me?! I did not see that coming at all.",
]

inputs = tokenizer(samples, truncation=True, padding=True, max_length=64, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model(**inputs).logits
preds = logits.argmax(dim=1).cpu().numpy()

for text, pred in zip(samples, preds):
    print(f"[{ID2LABEL[int(pred)]:>9}]  {text}")


## 8. Висновки

Результати повного запуску (Colab, GPU, повний датасет, тестова вибірка):

| Модель | Accuracy | Macro-F1 |
|---|---|---|
| TF-IDF + Logistic Regression | 0.9423 | 0.9179 |
| BiLSTM (з нуля) | 0.9604 | 0.9414 |
| DistilBERT (fine-tuned) | **0.9670** | **0.9480** |

Кожен крок ускладнення моделі покращив обидві метрики: transfer learning
(DistilBERT) > deep learning з нуля (BiLSTM) > класичний ML baseline.

**Висновки з confusion matrix (стабільні для всіх трьох моделей):**
- `fear` -> `surprise` — найстійкіша плутанина (7% baseline, 5% BiLSTM, 5%
  DistilBERT) — ймовірно, справжня неоднозначність розмітки, а не слабкість моделі.
- `joy` -> `love` витікає ~4% у кожній моделі — захоплені речення про `joy`
  і речення про `love` лексично близькі.
- Recall для `love` різко покращується з кращими репрезентаціями: 0.95
  (baseline) -> 0.99 (BiLSTM) -> 1.00 (DistilBERT).
- Неочевидний результат: recall DistilBERT для `fear` (0.89) *нижчий* за
  BiLSTM (0.93) — DistilBERT додає новий витік 3% `fear`->`sadness` поверх
  спільного витоку `fear`->`surprise`. DistilBERT виграє загалом, але не є
  рівномірно найкращим на кожному класі.

## 9. Можливі напрямки розвитку
- Аугментація даних / back-translation для рідкісних класів `surprise` та `love`.
- Спробувати більший трансформер (`bert-base`, `roberta-base`) або легший (`albert`, `tinybert`) для дослідження компромісу швидкість/точність.
- Ноутбук аналізу помилок: витягнути приклади з найбільшою похибкою і розглянути якісно — особливо межові випадки `fear`/`surprise` та `joy`/`love`.
- Обгорнути fine-tuned модель у невелике Gradio-демо для інтерактивного тестування.
